## **Step 1: Define the tools**

In [2]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
# This is raw model call. What we see in chat.openai.com has agent layer baked in as well.
llm = ChatOpenAI(model="gpt-5-mini") 

In [3]:
from langchain.tools import tool

In [4]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to gather information about current events or general knowledge"""

    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    
    response = search.invoke(query)
    return response

    
tool_duckduckgo_search.invoke("What is the capital of France?")


"France(/ˈfræns/ⓘor /ˈfrɑːns/; French pronunciation: [fʁɑ̃s]), officially the French Republic (French: République française, French pronunciation: [ʁepyblik fʁɑ̃sɛz]), is a country in Western Europe. It also includes various departments and territories ofFranceoverseas. MainlandFranceextends from the Mediterranean Sea to the English Channel and the North Sea, and from ... Where in the World is Paris found? Paris is thecapitalofFrance(French Republic), situated in the Western Europe subregion of Europe. In Paris, the currency used is Euro (€), which is the official currency used inFrance.TheLatitude, Longitude cordinates of Paris are 48.8534, 2.3488. WhatisthecapitalinFrance?Answer: ThecapitalofFranceisParis. Paris is not only the political and administrative center ofFrancebut also a vital cultural, economic, and historical hub. It is known worldwide for its iconic landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. ThecapitalofFranceisParis. Learn why this que

In [5]:
@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to gather information about historical events"""

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)
    return response
    
tool_wikipedia_search.invoke("Alan Turing")


'Page: Alan Turing\nSummary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.\nBorn in London, Turing was raised in southern England. He graduated from King\'s College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain\'s codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German ciphers, including improvement

In [6]:
@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to gather information about arXiv papers"""

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    #1. Initialize the ArxivAPIWrapper
    arxiv_api_wrapper = ArxivAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=4000
    )
    
    #2. Initialize the Arxiv Query Object
    arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)

    #3. Invoke the Arxiv Query Object
    response = arxiv.invoke(query)
    return response

tool_arxiv_search.invoke("What are the latest papers on AI?")

'Published: 2017-10-24\nTitle: Multi-messenger Observations of a Binary Neutron Star Merger\nAuthors: LIGO Scientific Collaboration, Virgo Collaboration, Fermi GBM, INTEGRAL, IceCube Collaboration, AstroSat Cadmium Zinc Telluride Imager Team, IPN Collaboration, The Insight-Hxmt Collaboration, ANTARES Collaboration, The Swift Collaboration, AGILE Team, The 1M2H Team, The Dark Energy Camera GW-EM Collaboration, the DES Collaboration, The DLT40 Collaboration, GRAWITA, :, GRAvitational Wave Inaf TeAm, The Fermi Large Area Telescope Collaboration, ATCA, :, Australia Telescope Compact Array, ASKAP, :, Australian SKA Pathfinder, Las Cumbres Observatory Group, OzGrav, DWF, AST3, CAASTRO Collaborations, The VINROUGE Collaboration, MASTER Collaboration, J-GEM, GROWTH, JAGWAR, Caltech- NRAO, TTU-NRAO, NuSTAR Collaborations, Pan-STARRS, The MAXI Team, TZAC Consortium, KU Collaboration, Nordic Optical Telescope, ePESSTO, GROND, Texas Tech University, SALT Group, TOROS, :, Transient Robotic Observat

In [7]:
@tool
def personal_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information. Expects a name as input."""
    
    infos = [
        {
            "name": "Ojas Dighe",
            "age": 25,
            "location": "India",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "John Doe",
            "age": 30,
            "location": "USA",
            "interests": "coding, building things, reading, writing"
        },
        {
            "name": "Jane Smith",
            "age": 28,
            "location": "Canada",
            "interests": "coding, building things, reading, writing"
        }
    ]

    for info in infos:
        if info["name"] == query:
            return f"{info['name']} is {info['age']} years old and lives in {info['location']}. {info['name']} likes {info['interests']}"
    return "No information found"

personal_info.invoke("John Doe")

'John Doe is 30 years old and lives in USA. John Doe likes coding, building things, reading, writing'

In [8]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool to answer questions based on NovaSphere organization data"""

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings

    embed_model = OpenAIEmbeddings(model = 'text-embedding-3-small')

    chroma_db_conn = Chroma(
        embedding_function = embed_model,
        persist_directory = '../CH-4_Agent/vector_db_semantic'
    )
    #1. Retrieve the most relevant chunks from the vector database
    relevant_docs = chroma_db_conn.similarity_search(query, k=3)

    #2. Create a string of the most relevant chunks
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])

    return relevant_docs_content
    
tool_rag.invoke("When was NovaSphere founded?")

/var/folders/25/h441spjj4nl4qg_jjtw1yxhh0000gn/T/ipykernel_45608/2429930526.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_conn = Chroma(


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

## Bind Tools

In [9]:
toolkit = [
            tool_duckduckgo_search, 
            tool_wikipedia_search, 
            tool_arxiv_search, 
            personal_info,
            tool_rag
          ]

# Step 1: Tool Binding
llm_bind = llm.bind_tools(toolkit)

llm_bind.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 258, 'total_tokens': 274, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXtD0sQH5TvnbtpWRZdVUC4V6HAP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8c21-3e22-75f0-bc8b-760ce9bc7f33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 258, 'output_tokens': 16, 'total_tokens': 274, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
# We will not be able to get content in LLM response even after tool binding. 
# You will see tool calls in the response, but no content in AIMessage Object.
# To solve this, we need to use a tool calling agent.
llm_bind.invoke("Tell me about Ojas Dighe. Make tool calls if necessary")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 264, 'total_tokens': 294, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXtFCzALtUVf7YkGxyNDcRrQ3W9m', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c21-4885-7942-8fc6-4dbb816c4bfb-0', tool_calls=[{'name': 'tool_duckduckgo_search', 'args': {'query': 'Ojas Dighe'}, 'id': 'call_VOUqw9gQptjHrNwVBy0LgUhn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 264, 'output_tokens': 30, 'total_tokens': 294, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## **ReAct Agent**

In [11]:
from langchain.agents import create_agent

agent = create_agent(llm_bind,toolkit)

In [12]:
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the age of John Doe? Make tool calls if necessary"}]}
)

{'messages': [HumanMessage(content='What is the age of John Doe? Make tool calls if necessary', additional_kwargs={}, response_metadata={}, id='932f20a8-9e58-42d6-98cd-400f8c2a8952'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 264, 'total_tokens': 352, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXtIQBJHLHeYA1Gf6esCHAqwG5Dc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c21-5362-7913-95ba-cd7040629a73-0', tool_calls=[{'name': 'personal_info', 'args': {'query': 'John Doe'}, 'id': 'call_CEhjfqz1AIFV7cXsnAjzXPds', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

In [13]:
# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere founded?"}]}
)

{'messages': [HumanMessage(content='When was NovaSphere founded?', additional_kwargs={}, response_metadata={}, id='59332f16-c11c-43b8-8ea8-1168041bd842'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 257, 'total_tokens': 286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUXtPxxvkMZmnNXQrdy8CEfLJIHQ3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c21-70bf-7253-bdc0-53376693892e-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere founded?'}, 'id': 'call_P0dhtfkT0s91QUlv0th798Ig', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2

## **Memory**

In [ ]:
agent_memory = []

In [36]:
def invoke_agent_with_memory(query: str):
    response = agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer the following query: {query}. You can use the following memory: {str(agent_memory)}"}]}
    )
    agent_memory.append({
        "query": query,
        "response": response
    })
    return response

invoke_agent_with_memory("When was NovaSphere founded?")

{'messages': [HumanMessage(content='Answer the following query: When was NovaSphere founded?. You can use the following memory: []', additional_kwargs={}, response_metadata={}, id='9c36f469-9f51-4d5f-a6cd-d5f5bd3a8add'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 270, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUY68JKfG32OwDRCWh1qbR1hM1Zuc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c2d-7b00-7c01-a177-5b0c6902fa00-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere founded?'}, 'id': 'call_1om2r1B9oRaWpfdNwF5ktI84', 'type': 'too

In [38]:
# Run the agent with memory
invoke_agent_with_memory("What is the capital of France?")

{'messages': [HumanMessage(content="Answer the following query: What is the capital of France?. You can use the following memory: [{'query': 'When was NovaSphere founded?', 'response': {'messages': [HumanMessage(content='Answer the following query: When was NovaSphere founded?. You can use the following memory: []', additional_kwargs={}, response_metadata={}, id='9c36f469-9f51-4d5f-a6cd-d5f5bd3a8add'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 270, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUY68JKfG32OwDRCWh1qbR1hM1Zuc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='l

In [39]:
invoke_agent_with_memory("Summarize the conversation so far")

{'messages': [HumanMessage(content='Answer the following query: Summarize the conversation so far. You can use the following memory: [{\'query\': \'When was NovaSphere founded?\', \'response\': {\'messages\': [HumanMessage(content=\'Answer the following query: When was NovaSphere founded?. You can use the following memory: []\', additional_kwargs={}, response_metadata={}, id=\'9c36f469-9f51-4d5f-a6cd-d5f5bd3a8add\'), AIMessage(content=\'\', additional_kwargs={\'refusal\': None}, response_metadata={\'token_usage\': {\'completion_tokens\': 93, \'prompt_tokens\': 270, \'total_tokens\': 363, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 64, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_provider\': \'openai\', \'model_name\': \'gpt-5-mini-2025-08-07\', \'system_fingerprint\': None, \'id\': \'chatcmpl-DUY68JKfG32OwDRCWh1qbR1hM1Zuc\', \'service_tier\': \'defau

In [40]:
agent_memory

[{'query': 'When was NovaSphere founded?',
  'response': {'messages': [HumanMessage(content='Answer the following query: When was NovaSphere founded?. You can use the following memory: []', additional_kwargs={}, response_metadata={}, id='9c36f469-9f51-4d5f-a6cd-d5f5bd3a8add'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 270, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUY68JKfG32OwDRCWh1qbR1hM1Zuc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8c2d-7b00-7c01-a177-5b0c6902fa00-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere foun

In [ ]:
while True:
    query = input("Ask me anything: ")
    response = invoke_agent_with_memory(query)
    print(response['messages'][-1].content)
    flag = input("Do you want to continue? (y/n): ")
    if flag == "n":
        break


Hi Ojas — nice to meet you. Based on the memory you gave me:

- You are 25 years old.
- You live in India.
- You enjoy coding, building things, reading, and writing.

Short bio: "Ojas Dighe is a 25-year-old from India who enjoys coding, building things, reading, and writing."

Would you like me to:
- Edit or add anything to that?
- Save these details as a memory so I remember them in future chats?
- Help turn your interests into project ideas, a learning plan, or a portfolio/bio for profiles?

Tell me what you’d like next.
NovaSphere (NovaSphere Technologies) is a fictional data-and-technology company created to represent a modern data engineering/analytics firm. Key points:

- Founded in 2016 by a small group of software engineers.  
- Core services: data engineering and analytics, helping organizations turn data into better decisions.  
- Industries served: finance, healthcare, retail, e‑commerce (and similar sectors).  
- Values: continuous learning, high quality of work, and helpin